In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore') 

Load data

In [2]:
ELEC_DATA_PATH = '../../data/raw/data/meters/cleaned/electricity_cleaned.csv'
WEATHER_DATA_PATH = '../../data/raw/data/weather/weather.csv'
METADATA_PATH = '../../data/raw/data/metadata/metadata.csv'

print("Loading data...")
elec_df = pd.read_csv(ELEC_DATA_PATH)
weather_df = pd.read_csv(WEATHER_DATA_PATH)
meta_df = pd.read_csv(METADATA_PATH)

print(f"Electricity data shape: {elec_df.shape}")
print(f"Weather data shape: {weather_df.shape}")

Loading data...
Electricity data shape: (17544, 1579)
Weather data shape: (331166, 10)


Isolate a Single Building and Merge Context

In [3]:
# 1. Isolate the target building
target_building = 'Panther_office_Hannah'
building_data = elec_df[['timestamp', target_building]].copy()
building_data.rename(columns={target_building: 'electricity_meter'}, inplace=True)

# 2. Find which site this building belongs to using the metadata
site_id = meta_df[meta_df['building_id'] == target_building]['site_id'].values[0]
print(f"Building {target_building} belongs to site: {site_id}")

# 3. Isolate the weather data for that specific site
site_weather = weather_df[weather_df['site_id'] == site_id].copy()

# 4. Merge the historical electricity usage with the weather context
merged_data = pd.merge(building_data, site_weather, on='timestamp', how='inner')
merged_data.head() 

Building Panther_office_Hannah belongs to site: Panther


,timestamp,electricity_meter,site_id,airTemperature,cloudCoverage,dewTemperature,precipDepth1HR,precipDepth6HR,seaLvlPressure,windDirection,windSpeed
0,2016-01-01 00:00:00,NaN,Panther,19.4,NaN,19.4,0.0,NaN,NaN,0.0,0.0
1,2016-01-01 01:00:00,NaN,Panther,21.1,6.0,21.1,-1.0,NaN,1019.4,0.0,0.0
2,2016-01-01 02:00:00,NaN,Panther,21.1,NaN,21.1,0.0,NaN,1018.8,210.0,1.5
3,2016-01-01 03:00:00,NaN,Panther,20.6,NaN,20.0,0.0,NaN,1018.1,0.0,0.0
4,2016-01-01 04:00:00,NaN,Panther,21.1,NaN,20.6,0.0,NaN,1019.0,290.0,1.5


Feature Engineering

In [4]:
# Convert timestamp string to datetime object
merged_data['timestamp'] = pd.to_datetime(merged_data['timestamp'])

# Extract time-based features
merged_data['hour'] = merged_data['timestamp'].dt.hour
merged_data['day_of_week'] = merged_data['timestamp'].dt.dayofweek
merged_data['month'] = merged_data['timestamp'].dt.month
merged_data['is_weekend'] = merged_data['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# Drop missing values 
merged_data = merged_data.dropna()

print("Feature engineering complete. Available columns:")
print(merged_data.columns.tolist())

Feature engineering complete. Available columns:
['timestamp', 'electricity_meter', 'site_id', 'airTemperature', 'cloudCoverage', 'dewTemperature', 'precipDepth1HR', 'precipDepth6HR', 'seaLvlPressure', 'windDirection', 'windSpeed', 'hour', 'day_of_week', 'month', 'is_weekend']


Train and Evaluate the Model

In [5]:
# 1. Select the features and target
features = ['hour', 'day_of_week', 'month', 'is_weekend', 'airTemperature', 'dewTemperature']
X = merged_data[features]
y = merged_data['electricity_meter']

# 2. Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and train the Random Forest
print("Training Random Forest baseline model...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1) 
model.fit(X_train, y_train)

# 4. Generate predictions on the unseen test data
predictions = model.predict(X_test)

# 5. Evaluate the model
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("\n--- Model Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

Training Random Forest baseline model...

--- Model Evaluation ---
Mean Absolute Error (MAE): 1.70
Root Mean Squared Error (RMSE): 2.21
